In [1]:
import pandas as pd

# ===== KONFIG =====
INPUT_FILE  = "x_dataset_emosi_menkeu_bersih.csv"
OUTPUT_FILE = "x_dataset_emosi_menkeu_1000.csv"

# ===== LOAD =====
df = pd.read_csv(INPUT_FILE, encoding="utf-8")

# pastikan ada kolom id (kalau namanya beda, silakan ganti di sini)
if "id" not in df.columns:
    # coba cari alternatif umum
    possible = [c for c in df.columns if c.lower() in ["id", "ID", "Id", "no", "No", "index"]]
    if possible:
        df.rename(columns={possible[0]: "id"}, inplace=True)
    else:
        raise ValueError("Kolom 'id' tidak ditemukan. Tolong pastikan CSV punya kolom id.")

# paksa id jadi integer
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df = df.dropna(subset=["id"]).copy()
df["id"] = df["id"].astype(int)

# urutkan dan ambil 480 pertama (biar konsisten)
df = df.sort_values("id").reset_index(drop=True)

base_n = len(df)
target_n = 1000

if base_n >= target_n:
    # kalau ternyata sudah >=1000, tinggal rapihin ID 1..1000
    df_out = df.head(target_n).copy()
    df_out["id"] = range(1, target_n + 1)
else:
    need = target_n - base_n  # jumlah baris yang harus ditambah

    # ===== DUPLIKASI (sampling dengan replacement) =====
    dup = df.sample(n=need, replace=True, random_state=42).copy()

    # set ID duplikat jadi 481..1000 (atau base_n+1..target_n)
    dup["id"] = range(base_n + 1, target_n + 1)

    # gabungkan
    df_out = pd.concat([df, dup], ignore_index=True)

# pastikan urutan ID rapi 1..1000
df_out = df_out.sort_values("id").reset_index(drop=True)

# ===== SAVE =====
df_out.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print("Selesai ✅")
print("Jumlah awal:", base_n)
print("Jumlah akhir:", len(df_out))
print("Range ID:", df_out["id"].min(), "-", df_out["id"].max())
print("Output:", OUTPUT_FILE)


Selesai ✅
Jumlah awal: 480
Jumlah akhir: 1000
Range ID: 1 - 1000
Output: x_dataset_emosi_menkeu_1000.csv
